In [ ]:
import torch

# Force CPU on Apple Silicon before importing Muograph
torch.backends.mps.is_available = lambda: False
torch.cuda.is_available = lambda: False

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from muograph.hits.hits import Hits
from muograph.tracking.tracking import Tracking, TrackingMST
from muograph.volume.volume import Volume
from muograph.reconstruction.binned_clustered import BCA
from muograph.utils.device import DEVICE

print("Muograph device:", DEVICE)

In [ ]:
DATA_DIR = Path(
    "../simulation/muon_scattering/results/"
    "benchmark_10000/spacing_400mm"
)

hits = pd.read_csv(DATA_DIR / "muon_hits.csv")
truth = pd.read_csv(DATA_DIR / "muon_truth.csv")
boundaries = pd.read_csv(DATA_DIR / "muon_boundaries.csv")

print("Generated muons:", len(truth))
print("Recorded detector hits:", len(hits))
print("Iron boundary records:", len(boundaries))

In [ ]:
# Require all six hits to lie within the active detector area
inside_area = (
    hits.x_mm.between(-300, 300)
    & hits.y_mm.between(-300, 300)
)

finite_hits = hits[inside_area].copy()

# Find muons detected on all six planes
counts = finite_hits.groupby("event_id").plane_id.nunique()
accepted_ids = counts[counts == 6].index

# Retain all six original hits for accepted muons
benchmark_hits = (
    hits[hits.event_id.isin(accepted_ids)]
    .sort_values(["event_id", "plane_id"])
    .reset_index(drop=True)
)

print("Generated muons:", len(truth))
print("Accepted tracks:", len(accepted_ids))
print("Accepted detector hits:", len(benchmark_hits))
print(f"Geometric acceptance: {100 * len(accepted_ids) / len(truth):.2f}%")

assert len(accepted_ids) == 1606
assert len(benchmark_hits) == 1606 * 6

In [ ]:
# Reproducible detector measurement noise
rng = np.random.default_rng(42)

benchmark_hits_noisy = benchmark_hits.copy()

benchmark_hits_noisy["x_mm"] += rng.normal(
    0.0, 1.0, len(benchmark_hits_noisy)
)

benchmark_hits_noisy["y_mm"] += rng.normal(
    0.0, 1.0, len(benchmark_hits_noisy)
)

print("Tracks:", benchmark_hits_noisy.event_id.nunique())
print("Hits:", len(benchmark_hits_noisy))

display(benchmark_hits_noisy.head(6))

In [ ]:
def make_muograph_hits(source, plane_ids):
    selected = source[
        source.plane_id.isin(plane_ids)
    ].copy()

    wide = selected.pivot(
        index="event_id",
        columns="plane_id",
        values=["x_mm", "y_mm", "z_mm"]
    )

    result = pd.DataFrame(index=wide.index)

    for plane in plane_ids:
        result[f"X{plane}"] = wide[("x_mm", plane)]
        result[f"Y{plane}"] = wide[("y_mm", plane)]
        result[f"Z{plane}"] = wide[("z_mm", plane)]

    return result.sort_index()


above_df = make_muograph_hits(
    benchmark_hits_noisy, [0, 1, 2]
)

below_df = make_muograph_hits(
    benchmark_hits_noisy, [3, 4, 5]
)

# Preserve original event IDs for later validation
event_order = above_df.index.to_numpy().copy()

# Muograph expects an energy column named E.
# Use initial kinetic energy in GeV as metadata.
energy_gev = (
    truth.set_index("event_id")
    .loc[event_order, "energy_true_MeV"]
    / 1000.0
)

above_df["E"] = energy_gev
below_df["E"] = energy_gev

assert above_df.index.equals(below_df.index)
assert len(above_df) == len(below_df) == 1606
assert not above_df.isna().any().any()
assert not below_df.isna().any().any()

print("Above shape:", above_df.shape)
print("Below shape:", below_df.shape)
print("Event IDs aligned:", above_df.index.equals(below_df.index))

display(above_df.head(3))

In [ ]:
hits_above = Hits(
    data=above_df,
    plane_labels=(0, 1, 2),
    spatial_res=(0.0, 0.0, 0.0),
    efficiency=1.0,
    input_unit="mm"
)

hits_below = Hits(
    data=below_df,
    plane_labels=(3, 4, 5),
    spatial_res=(0.0, 0.0, 0.0),
    efficiency=1.0,
    input_unit="mm"
)

tracking_above = Tracking(
    label="above",
    hits=hits_above
)

tracking_below = Tracking(
    label="below",
    hits=hits_below
)

tracking_mst = TrackingMST(
    trackings=(tracking_above, tracking_below)
)

voi = Volume(
    position=(0.0, 0.0, 0.0),
    dimension=(600.0, 600.0, 400.0),
    voxel_width=(100.0, 100.0, 100.0)
)

print("Upper tracks:", tracking_above.n_mu)
print("Lower tracks:", tracking_below.n_mu)
print("Combined MST tracks:", tracking_mst.n_mu)
print("Voxel grid:", voi.n_vox_xyz)

In [ ]:
def compute_numpy_angles(hits):
    rows = []

    for event_id, group in hits.groupby("event_id"):
        upper = group[group.plane_id.isin([0, 1, 2])]
        lower = group[group.plane_id.isin([3, 4, 5])]

        if len(upper) != 3 or len(lower) != 3:
            continue

        def fit_direction(data):
            ax, _ = np.polyfit(data.z_mm, data.x_mm, 1)
            ay, _ = np.polyfit(data.z_mm, data.y_mm, 1)

            direction = np.array([-ax, -ay, -1.0])
            return direction / np.linalg.norm(direction)

        vin = fit_direction(upper)
        vout = fit_direction(lower)

        theta = np.degrees(
            np.arccos(np.clip(np.dot(vin, vout), -1, 1))
        )

        rows.append({
            "event_id": event_id,
            "theta_numpy_deg": theta
        })

    return pd.DataFrame(rows)


numpy_angles = compute_numpy_angles(benchmark_hits_noisy)

print("Independent reconstructed tracks:", len(numpy_angles))
print("Median scattering angle:", numpy_angles.theta_numpy_deg.median())

In [ ]:
# Muograph stores scattering angles in radians
native_angles_deg = np.degrees(
    tracking_mst.dtheta.detach().cpu().numpy()
)

native_results = pd.DataFrame({
    "event_id": event_order,
    "theta_muograph_deg": native_angles_deg
})

angle_comparison = numpy_angles.merge(
    native_results,
    on="event_id",
    validate="one_to_one"
)

angle_comparison["difference_deg"] = (
    angle_comparison.theta_muograph_deg
    - angle_comparison.theta_numpy_deg
)

print("Compared tracks:", len(angle_comparison))

print(
    "Mean absolute difference:",
    angle_comparison.difference_deg.abs().mean(),
    "degrees"
)

print(
    "Maximum absolute difference:",
    angle_comparison.difference_deg.abs().max(),
    "degrees"
)

display(angle_comparison.head(10).round(6))

In [ ]:
print("Accepted muons:", tracking_mst.n_mu)
print("VOI dimensions:", voi.n_vox_xyz)
print("Mean scattering angle (deg):",
      np.degrees(tracking_mst.dtheta.mean().item()))
print("Maximum scattering angle (deg):",
      np.degrees(tracking_mst.dtheta.max().item()))

In [ ]:
# Native BCA reconstruction using the corrected Muograph code

bca = BCA(
    voi=voi,
    tracking=tracking_mst
)

# Use scattering angles without momentum weighting
bca.params.use_p = False

print("BCA initialized")
print("Minimum muons per voxel:", bca.params.n_min_per_vox)
print("Maximum muons per voxel:", bca.params.n_max_per_vox)
print("Momentum weighting:", bca.params.use_p)

In [ ]:
# Retrieve native BCA reconstruction scores
bca_scores = bca.xyz_voxel_pred

print("BCA score tensor shape:", bca_scores.shape)
print("Data type:", bca_scores.dtype)

# Convert to NumPy
bca_array = bca_scores.detach().cpu().numpy()

print("Finite voxels:", np.isfinite(bca_array).sum())
print("Total voxels:", bca_array.size)
print("Minimum score:", np.nanmin(bca_array))
print("Median score:", np.nanmedian(bca_array))
print("Maximum score:", np.nanmax(bca_array))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

vmin = np.nanmin(bca_array)
vmax = np.nanmax(bca_array)

for iz, ax in enumerate(axes):
    image = ax.imshow(
        bca_array[:, :, iz].T,
        origin="lower",
        extent=(-300, 300, -300, 300),
        cmap="inferno",
        vmin=vmin,
        vmax=vmax,
        interpolation="nearest",
    )

    z_center = -150 + iz * 100

    ax.set_title(f"z = {z_center} mm")
    ax.set_xlabel("x (mm)")
    ax.set_ylabel("y (mm)")
    ax.set_aspect("equal")

fig.colorbar(
    image,
    ax=axes,
    label="BCA reconstruction score",
    shrink=0.75
)

plt.show()

In [ ]:
from scipy.stats import spearmanr

# Voxel boundaries, in mm
x_edges = np.linspace(-300, 300, 7)
y_edges = np.linspace(-300, 300, 7)
z_edges = np.linspace(-200, 200, 5)

def overlap_fraction(edges, low, high):
    widths = np.diff(edges)
    overlap = np.maximum(
        0,
        np.minimum(edges[1:], high)
        - np.maximum(edges[:-1], low)
    )
    return overlap / widths

fx = overlap_fraction(x_edges, -250, 250)
fy = overlap_fraction(y_edges, -250, 250)
fz = overlap_fraction(z_edges, -100, 100)

iron_fraction = (
    fx[:, None, None]
    * fy[None, :, None]
    * fz[None, None, :]
)

rho, p_value = spearmanr(
    iron_fraction.ravel(),
    bca_array.ravel()
)

print("Iron fraction shape:", iron_fraction.shape)
print("Voxels containing iron:", np.count_nonzero(iron_fraction))
print("Spearman correlation:", rho)
print("p-value:", p_value)

print("\nMean BCA score:")
print("Iron-containing voxels:",
      bca_array[iron_fraction > 0].mean())
print("Air-only voxels:",
      bca_array[iron_fraction == 0].mean())

In [ ]:
print("BCA attributes:")
print([
    name for name in dir(bca)
    if any(term in name.lower() for term in [
        "poca", "voxel", "count", "hit"
    ])
])

In [ ]:
occupancy = bca.n_poca_per_vox

print("Occupancy type:", type(occupancy))
print("Occupancy shape:", occupancy.shape)

if isinstance(occupancy, torch.Tensor):
    occupancy = occupancy.detach().cpu().numpy()
else:
    occupancy = np.asarray(occupancy)

print("Minimum occupancy:", occupancy.min())
print("Maximum occupancy:", occupancy.max())
print("Occupied voxels:", np.count_nonzero(occupancy))
print("Voxels with >= 3 PoCA points:", np.sum(occupancy >= 3))
print("Voxels with >= 5 PoCA points:", np.sum(occupancy >= 5))
print("Voxels with >= 10 PoCA points:", np.sum(occupancy >= 10))
print("Total assigned PoCA points:", occupancy.sum())

In [ ]:
from scipy.stats import spearmanr

thresholds = [1, 3, 5, 10, 15, 20]

results = []

for threshold in thresholds:
    mask = (
        (occupancy >= threshold)
        & np.isfinite(bca_array)
    )

    n_voxels = int(mask.sum())

    if n_voxels < 3:
        continue

    rho, p_value = spearmanr(
        iron_fraction[mask],
        bca_array[mask]
    )

    results.append({
        "min_occupancy": threshold,
        "n_voxels": n_voxels,
        "bca_spearman": rho,
        "p_value": p_value
    })

bca_correlations = pd.DataFrame(results)

display(bca_correlations.round(4))

In [ ]:
for name in ["poca_indices", "poca_points", "bca_poca_points"]:
    value = getattr(bca, name)

    print(
        name,
        "shape:", getattr(value, "shape", None),
        "type:", type(value)
    )

print("Tracking events:", tracking_mst.n_mu)
print("Total voxel assignments:", occupancy.sum())

In [ ]:
print("BCA filtered PoCA points:", bca.poca_points.shape)
print("BCA voxel indices:", bca.poca_indices.shape)

print("\nTracking scattering angles:",
      tracking_mst.dtheta.shape)

print("\nBCA internal attributes:")
for name in sorted(vars(bca)):
    if any(term in name.lower() for term in [
        "mask", "theta", "poca", "index", "track"
    ]):
        value = getattr(bca, name)
        print(
            name,
            getattr(value, "shape", None),
            type(value).__name__
        )

In [ ]:
print("Original tracks:", tracking_mst.dtheta.shape)
print("BCA tracks:", bca.bca_tracks.dtheta.shape)

for name in ["_mask_in_voi", "_parallel_mask", "mask"]:
    m = getattr(bca, name)

    print(f"\n{name}")
    print("Shape:", tuple(m.shape))
    print("Dtype:", m.dtype)

    if m.dtype == torch.bool:
        print("True entries:", int(m.sum().item()))
        print("False entries:", int((~m).sum().item()))

print("\nAssigned PoCA points:", len(bca.poca_indices))

In [ ]:
# Matched PoCA mean(theta²) reconstruction
# Using exactly the same events and voxel assignments as BCA

indices = bca.poca_indices.detach().cpu().numpy().astype(int)

theta = (
    bca.bca_tracks.dtheta
    .detach()
    .cpu()
    .numpy()
)

assert len(indices) == len(theta) == 1081

# Scores use squared scattering angles in degrees
theta2 = np.degrees(theta) ** 2

score_sum = np.zeros((6, 6, 4), dtype=float)
point_count = np.zeros((6, 6, 4), dtype=int)

np.add.at(
    score_sum,
    (indices[:, 0], indices[:, 1], indices[:, 2]),
    theta2
)

np.add.at(
    point_count,
    (indices[:, 0], indices[:, 1], indices[:, 2]),
    1
)

poca_mean_theta2 = np.full((6, 6, 4), np.nan)

np.divide(
    score_sum,
    point_count,
    out=poca_mean_theta2,
    where=point_count > 0
)

print("Matched PoCA points:", point_count.sum())
print("Occupied voxels:", np.count_nonzero(point_count))

print(
    "Occupancy agrees with BCA:",
    np.array_equal(point_count, occupancy)
)

In [ ]:
thresholds = [1, 3, 5, 10, 15, 20]

results = []

for threshold in thresholds:
    mask = (
        (point_count >= threshold)
        & np.isfinite(poca_mean_theta2)
        & np.isfinite(bca_array)
    )

    poca_rho, _ = spearmanr(
        iron_fraction[mask],
        poca_mean_theta2[mask]
    )

    bca_rho, _ = spearmanr(
        iron_fraction[mask],
        bca_array[mask]
    )

    results.append({
        "min_occupancy": threshold,
        "n_voxels": int(mask.sum()),
        "poca_rho": poca_rho,
        "bca_rho": bca_rho,
        "difference_bca_minus_poca": bca_rho - poca_rho
    })

matched_comparison = pd.DataFrame(results)

display(matched_comparison.round(4))

In [ ]:
from scipy.stats import spearmanr

mask = occupancy >= 3

print("Voxels analyzed:", int(mask.sum()))

print(
    "Occupancy vs PoCA score:",
    spearmanr(
        occupancy[mask],
        poca_mean_theta2[mask]
    ).statistic
)

print(
    "Occupancy vs BCA score:",
    spearmanr(
        occupancy[mask],
        bca_array[mask]
    ).statistic
)

print(
    "Occupancy vs iron fraction:",
    spearmanr(
        occupancy[mask],
        iron_fraction[mask]
    ).statistic
)

In [ ]:
rows = []

for lower, upper in [(3, 5), (5, 10), (10, 15), (15, 100)]:
    mask = (occupancy >= lower) & (occupancy < upper)

    if mask.sum() < 3:
        continue

    rows.append({
        "occupancy_group": f"{lower}–{upper - 1}",
        "n_voxels": int(mask.sum()),
        "mean_iron_fraction": iron_fraction[mask].mean(),
        "mean_poca_score": poca_mean_theta2[mask].mean(),
        "mean_bca_score": bca_array[mask].mean(),
        "bca_score_std": bca_array[mask].std(),
    })

occupancy_groups = pd.DataFrame(rows)

display(occupancy_groups.round(4))

In [ ]:

# BCA ablation: effect of pairwise-distance weighting

from scipy.stats import spearmanr

# Use the actual events retained by native BCA
idx = bca.bca_indices.detach().cpu().numpy().astype(int)
points = bca.bca_poca_points.detach().cpu().numpy()

theta_tensor = bca.bca_tracks.dtheta.detach().cpu()

# Reproduce Muograph's angle clamping
theta_max = torch.quantile(
    theta_tensor,
    q=bca.params.dtheta_clamp
)
theta = torch.clamp(theta_tensor, max=theta_max).numpy()

shape = tuple(voi.n_vox_xyz)

angle_only = np.full(shape, np.nan)
distance_weighted = np.full(shape, np.nan)

for ix in range(shape[0]):
    for iy in range(shape[1]):
        for iz in range(shape[2]):

            selected = np.all(
                idx == [ix, iy, iz],
                axis=1
            )

            if selected.sum() < 3:
                continue

            t = theta[selected]
            p = points[selected]

            # Unique pairs, excluding diagonal
            a, b = np.tril_indices(len(t), k=-1)

            weights = t[a] * t[b]
            distances = np.linalg.norm(p[a] - p[b], axis=1)

            valid = (
                (weights > 0)
                & (distances > 0)
            )

            if not np.any(valid):
                continue

            angle_only[ix, iy, iz] = np.median(
                np.log(weights[valid])
            )

            distance_weighted[ix, iy, iz] = np.median(
                np.log(weights[valid] / distances[valid])
            )

# Compare only voxels with valid scores
rows = []

for threshold in [3, 5, 10, 15, 20]:
    mask = (
        (occupancy >= threshold)
        & np.isfinite(angle_only)
        & np.isfinite(distance_weighted)
    )

    if mask.sum() < 3:
        continue

    rows.append({
        "min_occupancy": threshold,
        "n_voxels": int(mask.sum()),
        "poca_rho": spearmanr(
            iron_fraction[mask],
            poca_mean_theta2[mask]
        ).statistic,
        "angle_only_rho": spearmanr(
            iron_fraction[mask],
            angle_only[mask]
        ).statistic,
        "distance_weighted_rho": spearmanr(
            iron_fraction[mask],
            distance_weighted[mask]
        ).statistic,
        "native_bca_rho": spearmanr(
            iron_fraction[mask],
            bca_array[mask]
        ).statistic,
    })

ablation_results = pd.DataFrame(rows)

display(ablation_results.round(4))


In [ ]:

# Aggregation ablation using matched PoCA events

median_theta2 = np.full(shape, np.nan)
mean_log_theta2 = np.full(shape, np.nan)

theta_deg = np.degrees(theta)

for ix in range(shape[0]):
    for iy in range(shape[1]):
        for iz in range(shape[2]):

            selected = np.all(idx == [ix, iy, iz], axis=1)

            if selected.sum() < 3:
                continue

            t2 = theta_deg[selected] ** 2

            median_theta2[ix, iy, iz] = np.median(t2)

            positive = t2[t2 > 0]
            if len(positive):
                mean_log_theta2[ix, iy, iz] = np.mean(
                    np.log(positive)
                )

rows = []

for threshold in [3, 5, 10, 15]:
    mask = (
        (occupancy >= threshold)
        & np.isfinite(median_theta2)
        & np.isfinite(mean_log_theta2)
        & np.isfinite(angle_only)
    )

    rows.append({
        "min_occupancy": threshold,
        "n_voxels": int(mask.sum()),
        "mean_theta2": spearmanr(
            iron_fraction[mask],
            poca_mean_theta2[mask]
        ).statistic,
        "median_theta2": spearmanr(
            iron_fraction[mask],
            median_theta2[mask]
        ).statistic,
        "mean_log_theta2": spearmanr(
            iron_fraction[mask],
            mean_log_theta2[mask]
        ).statistic,
        "pairwise_median_log": spearmanr(
            iron_fraction[mask],
            angle_only[mask]
        ).statistic,
    })

aggregation_results = pd.DataFrame(rows)

display(aggregation_results.round(4))


In [ ]:
# Mean theta² using exactly the angle preprocessing used by BCA

mean_theta2_clamped = np.full(shape, np.nan)

theta_deg_clamped = np.degrees(theta)

for ix in range(shape[0]):
    for iy in range(shape[1]):
        for iz in range(shape[2]):
            selected = np.all(idx == [ix, iy, iz], axis=1)

            if selected.sum() == 0:
                continue

            mean_theta2_clamped[ix, iy, iz] = np.mean(
                theta_deg_clamped[selected] ** 2
            )

rows = []

for threshold in [3, 5, 10, 15]:
    mask = (
        (occupancy >= threshold)
        & np.isfinite(mean_theta2_clamped)
        & np.isfinite(poca_mean_theta2)
    )

    rho_original = spearmanr(
        iron_fraction[mask],
        poca_mean_theta2[mask]
    ).statistic

    rho_clamped = spearmanr(
        iron_fraction[mask],
        mean_theta2_clamped[mask]
    ).statistic

    rows.append({
        "min_occupancy": threshold,
        "n_voxels": int(mask.sum()),
        "original_mean_theta2": rho_original,
        "clamped_mean_theta2": rho_clamped,
        "difference": rho_clamped - rho_original,
    })

clamping_results = pd.DataFrame(rows)
display(clamping_results.round(4))

print(
    "Events affected by clamping:",
    int(np.sum(
        bca.bca_tracks.dtheta.detach().cpu().numpy()
        > theta_max.item()
    ))
)

In [ ]:
# Compare scattering-angle distributions by assigned voxel material

event_iron_fraction = iron_fraction[
    idx[:, 0],
    idx[:, 1],
    idx[:, 2]
]

theta_deg_raw = np.degrees(
    bca.bca_tracks.dtheta.detach().cpu().numpy()
)

iron_angles = theta_deg_raw[event_iron_fraction > 0]
air_angles = theta_deg_raw[event_iron_fraction == 0]

summary = pd.DataFrame({
    "Metric": [
        "Number of muons",
        "Median angle (deg)",
        "Mean angle (deg)",
        "90th percentile (deg)",
        "95th percentile (deg)",
        "Fraction above 1°",
        "Fraction above 2°"
    ],
    "Iron-containing voxel": [
        len(iron_angles),
        np.median(iron_angles),
        np.mean(iron_angles),
        np.quantile(iron_angles, 0.90),
        np.quantile(iron_angles, 0.95),
        np.mean(iron_angles > 1),
        np.mean(iron_angles > 2)
    ],
    "Air-only voxel": [
        len(air_angles),
        np.median(air_angles),
        np.mean(air_angles),
        np.quantile(air_angles, 0.90),
        np.quantile(air_angles, 0.95),
        np.mean(air_angles > 1),
        np.mean(air_angles > 2)
    ]
})

display(summary.round(4))

In [ ]:
# Test how sensitive PoCA performance is to large scattering angles

theta_deg_raw = np.degrees(
    bca.bca_tracks.dtheta.detach().cpu().numpy()
)

caps = [None, 2.0, 1.5, 1.0, 0.75]

rows = []

for cap in caps:
    if cap is None:
        theta_used = theta_deg_raw.copy()
        label = "No cap"
    else:
        theta_used = np.minimum(theta_deg_raw, cap)
        label = f"{cap}°"

    scores = np.full(shape, np.nan)

    for ix in range(shape[0]):
        for iy in range(shape[1]):
            for iz in range(shape[2]):
                selected = np.all(
                    idx == [ix, iy, iz],
                    axis=1
                )

                if not selected.any():
                    continue

                scores[ix, iy, iz] = np.mean(
                    theta_used[selected] ** 2
                )

    for threshold in [3, 5, 10, 15]:
        mask = (
            (occupancy >= threshold)
            & np.isfinite(scores)
        )

        rho = spearmanr(
            iron_fraction[mask],
            scores[mask]
        ).statistic

        rows.append({
            "angle_cap": label,
            "min_occupancy": threshold,
            "n_voxels": int(mask.sum()),
            "spearman": rho
        })

cap_results = pd.DataFrame(rows)

display(
    cap_results.pivot(
        index="angle_cap",
        columns="min_occupancy",
        values="spearman"
    ).round(4)
)